# Audiencias de Marketing — Laboratorio Económico

Un solo notebook para comprender las audiencias de Medallio con mirada de economista: tamaño de mercado, composición, cohortes, matrices, incertidumbre, hipótesis y diseño experimental.

> Privacidad: este cuaderno no extrae nombres, documentos, teléfonos, correos ni `persona_key`. Trabaja exclusivamente con variables analíticas sin PII.

## Marco económico

- **Universo:** personas registradas en el CRM.
- **Resultado proxy:** avance a proforma, separación o minuta/venta.
- **Oportunidad RMKT:** personas antiguas sin avance y con contacto válido.
- **Restricción institucional:** una persona solo es entregable externamente con consentimiento verificable y sin supresión.
- **Advertencia:** las diferencias observadas entre canales son asociación, no causalidad. La causalidad requiere asignación aleatoria o un diseño cuasiexperimental defendible.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest, proportion_confint

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
plt.style.use('seaborn-v0_8-whitegrid')

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Abra el notebook dentro del repositorio bd_replica_crm.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from replica_cygnus.connections import connect_postgres
from replica_cygnus.settings import load_settings

PROJECT_ROOT

## 1. Extracción local y contrato de datos

La consulta es de solo lectura y usa PostgreSQL local. La unidad estadística es una persona deduplicada por el mart.

In [ ]:
AUDIENCE_SQL = '''
SELECT
    COALESCE(NULLIF(proyecto, ''), 'Sin proyecto') AS proyecto,
    COALESCE(NULLIF(medio_captacion, ''), 'Sin medio') AS medio_captacion,
    fecha_cliente,
    fecha_ultimo_avance,
    etapa_maxima,
    positiva::int AS positiva,
    negativa_sin_avance::int AS negativa_sin_avance,
    (exclusion_convertido OR exclusion_contacto_invalido OR exclusion_no_contactar)::int AS exclusion,
    contacto_valido::int AS contacto_valido,
    consentimiento_publicidad_verificado::int AS consentimiento_verificado,
    apto_entrega_externa::int AS apto_entrega_externa
FROM analytics.marketing_audiencias
'''

HEALTH_SQL = 'SELECT * FROM analytics.v_marketing_audiencias_health'

def query_df(sql: str) -> pd.DataFrame:
    with connect_postgres(load_settings(require_source=False)) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute('SET TRANSACTION READ ONLY')
                cur.execute(sql)
                columns = [column.name for column in cur.description]
                return pd.DataFrame(cur.fetchall(), columns=columns)

aud = query_df(AUDIENCE_SQL)
health = query_df(HEALTH_SQL)
for column in ['fecha_cliente', 'fecha_ultimo_avance']:
    aud[column] = pd.to_datetime(aud[column], errors='coerce', utc=True).dt.tz_localize(None)

assert len(aud) == int(health.loc[0, 'personas']), 'El detalle no coincide con la vista health.'
display(health)
print(f'Filas analíticas: {len(aud):,} | Columnas: {aud.shape[1]} | PII extraída: NO')

## 2. Diccionario económico

| Variable | Lectura |
|---|---|
| `positiva` | Algún avance: proforma, separación o minuta/venta |
| `negativa_sin_avance` | Sin avance y antigüedad igual o mayor a 180 días |
| `contacto_valido` | Existe teléfono o correo que pasó controles de calidad |
| `exclusion` | Conversión, contacto inválido o supresión explícita |
| `apto_entrega_externa` | Contacto válido + consentimiento vigente + sin supresión |
| `etapa_maxima` | Máximo estado comercial observado |

## 3. Estadística descriptiva y composición

In [ ]:
kpis = pd.Series({
    'Personas': len(aud),
    'Tasa positiva': aud['positiva'].mean(),
    'Tasa sin avance': aud['negativa_sin_avance'].mean(),
    'Contactabilidad': aud['contacto_valido'].mean(),
    'Tasa de exclusión': aud['exclusion'].mean(),
    'Aptitud para entrega externa': aud['apto_entrega_externa'].mean(),
}, name='valor')
display(kpis.to_frame().style.format({'valor': lambda x: f'{x:,.0f}' if x > 1 else f'{x:.1%}'}))

stage_order = ['sin_avance', 'proforma', 'separacion', 'minuta_venta']
stage = aud['etapa_maxima'].value_counts().reindex(stage_order, fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(stage.index, stage.values, color=['#94a3b8', '#38bdf8', '#f59e0b', '#10b981'])
ax.bar_label(bars, labels=[f'{v:,.0f}' for v in stage.values], padding=3)
ax.set(title='Composición del universo por máxima etapa', ylabel='Personas', xlabel='Etapa')
plt.show()

## 4. Matriz proyecto × etapa

Los conteos muestran escala; los porcentajes por fila muestran composición. Conviene leer ambos: un porcentaje alto en una base pequeña puede no ser una prioridad económica.

In [ ]:
top_projects = aud['proyecto'].value_counts().head(15).index
project_stage_n = pd.crosstab(aud['proyecto'], aud['etapa_maxima']).reindex(columns=stage_order, fill_value=0).loc[top_projects]
project_stage_pct = project_stage_n.div(project_stage_n.sum(axis=1), axis=0)
display(project_stage_n.style.format('{:,.0f}').background_gradient(cmap='Blues', axis=None))
display(project_stage_pct.style.format('{:.1%}').background_gradient(cmap='YlGn', axis=None))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
image = ax.imshow(project_stage_pct.values, aspect='auto', cmap='YlGn', vmin=0, vmax=max(0.01, project_stage_pct.values.max()))
ax.set_xticks(range(len(project_stage_pct.columns)), project_stage_pct.columns, rotation=30, ha='right')
ax.set_yticks(range(len(project_stage_pct.index)), project_stage_pct.index)
ax.set_title('Matriz de composición por proyecto')
fig.colorbar(image, ax=ax, label='Participación dentro del proyecto')
plt.tight_layout()
plt.show()

## 5. Canales y concentración

El índice HHI resume concentración: valores más altos significan mayor dependencia de pocos canales.

In [ ]:
channel = aud.groupby('medio_captacion', observed=True).agg(
    personas=('positiva', 'size'),
    positivas=('positiva', 'sum'),
    tasa_positiva=('positiva', 'mean'),
    negativas_rmkt=('negativa_sin_avance', 'sum'),
    contactabilidad=('contacto_valido', 'mean'),
).sort_values('personas', ascending=False)
channel['participacion'] = channel['personas'] / channel['personas'].sum()
hhi = float((channel['participacion'] ** 2).sum())
display(channel.head(20).style.format({
    'personas': '{:,.0f}', 'positivas': '{:,.0f}', 'negativas_rmkt': '{:,.0f}',
    'tasa_positiva': '{:.1%}', 'contactabilidad': '{:.1%}', 'participacion': '{:.1%}'
}))
print(f'HHI de captación: {hhi:.3f}')

## 6. Cohortes

Una cohorte agrupa personas por mes de creación. Las cohortes recientes tienen menos tiempo para avanzar, por lo que una comparación directa puede sufrir censura temporal.

In [ ]:
cohort = aud.dropna(subset=['fecha_cliente']).copy()
cohort['cohorte'] = cohort['fecha_cliente'].dt.to_period('M').dt.to_timestamp()
cohort_month = cohort.groupby('cohorte').agg(
    personas=('positiva', 'size'),
    tasa_positiva=('positiva', 'mean'),
    tasa_sin_avance=('negativa_sin_avance', 'mean'),
    contactabilidad=('contacto_valido', 'mean'),
).sort_index()
recent_cohorts = cohort_month.tail(36)
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(recent_cohorts.index, recent_cohorts['personas'], width=20, alpha=.25, color='#64748b', label='Personas')
ax1.set_ylabel('Personas')
ax2 = ax1.twinx()
ax2.plot(recent_cohorts.index, recent_cohorts['tasa_positiva'], color='#0284c7', marker='o', label='Tasa positiva')
ax2.set_ylabel('Tasa positiva')
ax2.set_ylim(0, max(.05, recent_cohorts['tasa_positiva'].max() * 1.2))
ax1.set_title('Cohortes: escala y tasa de avance proxy')
fig.autofmt_xdate()
plt.show()

## 7. Incertidumbre: intervalos de confianza

Un ranking puntual puede exagerar diferencias. El intervalo Wilson al 95% muestra la precisión de cada tasa de avance.

In [ ]:
project_stats = aud.groupby('proyecto').agg(n=('positiva', 'size'), exitos=('positiva', 'sum')).query('n >= 100').copy()
project_stats['tasa'] = project_stats['exitos'] / project_stats['n']
ci = project_stats.apply(lambda r: proportion_confint(r['exitos'], r['n'], alpha=.05, method='wilson'), axis=1)
project_stats[['ci_low', 'ci_high']] = pd.DataFrame(ci.tolist(), index=project_stats.index)
ranked = project_stats.sort_values('tasa', ascending=False).head(15).sort_values('tasa')
fig, ax = plt.subplots(figsize=(9, 7))
errors = np.vstack([ranked['tasa'] - ranked['ci_low'], ranked['ci_high'] - ranked['tasa']])
ax.errorbar(ranked['tasa'], ranked.index, xerr=errors, fmt='o', color='#0369a1', ecolor='#7dd3fc', capsize=3)
ax.set(title='Tasa positiva e intervalo Wilson 95%', xlabel='Tasa de avance proxy', ylabel='Proyecto')
plt.show()

## 8. Hipótesis 1 — ¿Canal y avance son independientes?

$H_0$: la condición positiva es independiente del canal. $H_1$: existe asociación. Cramér V aproxima la magnitud de esa asociación.

In [ ]:
eligible_channels = aud['medio_captacion'].value_counts()
eligible_channels = eligible_channels[eligible_channels >= 100].index
hyp = aud[aud['medio_captacion'].isin(eligible_channels)]
contingency = pd.crosstab(hyp['medio_captacion'], hyp['positiva'])
chi2, p_value, dof, expected = chi2_contingency(contingency)
n = contingency.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * max(1, min(contingency.shape) - 1)))
decision = 'rechazar H0: hay asociación estadística' if p_value < .05 else 'no rechazar H0'
display(contingency)
display(Markdown(f'**Resultado:** χ²={chi2:,.2f}, gl={dof}, p={p_value:.3g}, Cramér V={cramers_v:.3f}. Se decide **{decision}**. La significancia no demuestra causalidad ni garantiza importancia económica.'))

## 9. Hipótesis 2 — comparación de los dos canales de mayor escala

Se contrasta igualdad de proporciones y se reporta diferencia con intervalo al 95%. La selección automática evita asumir nombres de canales.

In [ ]:
top_two = channel.head(2).index.tolist()
if len(top_two) == 2:
    a, b = top_two
    sample = aud[aud['medio_captacion'].isin(top_two)]
    summary = sample.groupby('medio_captacion')['positiva'].agg(['sum', 'count', 'mean']).loc[top_two]
    z_stat, p_two = proportions_ztest(summary['sum'].to_numpy(), summary['count'].to_numpy())
    low, high = confint_proportions_2indep(summary.loc[a, 'sum'], summary.loc[a, 'count'], summary.loc[b, 'sum'], summary.loc[b, 'count'], method='wald')
    diff = summary.loc[a, 'mean'] - summary.loc[b, 'mean']
    display(summary.style.format({'sum': '{:,.0f}', 'count': '{:,.0f}', 'mean': '{:.1%}'}))
    display(Markdown(f'**{a} − {b}:** diferencia={diff:.2%}; IC95%=[{low:.2%}, {high:.2%}]; z={z_stat:.2f}; p={p_two:.3g}. Interpretar como brecha descriptiva ajustada por incertidumbre, no como efecto del canal.'))
else:
    print('No existen dos canales con información suficiente.')

## 10. Matriz de oportunidad RMKT

El eje horizontal mide contactabilidad; el vertical, personas sin avance con contacto válido. Los cuadrantes son una herramienta de priorización, no una autorización de campaña.

In [ ]:
opportunity = aud.assign(negativa_contactable=lambda x: x['negativa_sin_avance'] * x['contacto_valido']).groupby('proyecto').agg(
    personas=('positiva', 'size'),
    negativas_contactables=('negativa_contactable', 'sum'),
    contactabilidad=('contacto_valido', 'mean'),
    tasa_positiva=('positiva', 'mean'),
).query('personas >= 50').sort_values('negativas_contactables', ascending=False)
fig, ax = plt.subplots(figsize=(11, 7))
sizes = 40 + 500 * opportunity['personas'] / opportunity['personas'].max()
ax.scatter(opportunity['contactabilidad'], opportunity['negativas_contactables'], s=sizes, alpha=.55, color='#7c3aed')
ax.axvline(opportunity['contactabilidad'].median(), color='#94a3b8', linestyle='--')
ax.axhline(opportunity['negativas_contactables'].median(), color='#94a3b8', linestyle='--')
for project, row in opportunity.head(12).iterrows():
    ax.annotate(project, (row['contactabilidad'], row['negativas_contactables']), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set(title='Matriz de oportunidad de remarketing por proyecto', xlabel='Contactabilidad', ylabel='Personas antiguas sin avance y contactables')
plt.show()
display(opportunity.head(20).style.format({'personas': '{:,.0f}', 'negativas_contactables': '{:,.0f}', 'contactabilidad': '{:.1%}', 'tasa_positiva': '{:.1%}'}))

## 11. Gate de privacidad y factibilidad operativa

In [ ]:
aptas = int(aud['apto_entrega_externa'].sum())
consentidas = int(aud['consentimiento_verificado'].sum())
if aptas == 0:
    display(Markdown(f'### ⛔ Análisis habilitado; activación externa bloqueada — Consentimientos verificados: **{consentidas:,}**. Personas aptas para entrega externa: **0**. Antes de exportar a una plataforma publicitaria debe cargarse evidencia autorizada en `analytics.marketing_consentimientos`.'))
else:
    display(Markdown(f'### Gate operativo — Hay **{aptas:,}** personas aptas para entrega externa. Toda exportación debe mantener el filtro `apto_entrega_externa = true`.'))

## 12. Diseño experimental y potencia

Propuesta: dentro de la audiencia RMKT apta, asignar aleatoriamente tratamiento y grupo control. Resultado primario: avance dentro de una ventana predefinida. El cálculo siguiente es ilustrativo y debe reemplazarse con la tasa base y el efecto mínimo económicamente relevante acordados.

In [ ]:
P_BASE = 0.02
P_TRATAMIENTO = 0.03
ALPHA = 0.05
POWER = 0.80
effect_size = 2 * (np.arcsin(np.sqrt(P_TRATAMIENTO)) - np.arcsin(np.sqrt(P_BASE)))
n_group = int(np.ceil(NormalIndPower().solve_power(effect_size=effect_size, alpha=ALPHA, power=POWER, ratio=1)))
display(Markdown(f'Con tasa base **{P_BASE:.1%}**, meta **{P_TRATAMIENTO:.1%}**, α={ALPHA:.0%} y potencia={POWER:.0%}, se requieren aproximadamente **{n_group:,} personas por grupo**. El KPI económico complementario debe ser margen incremental neto del costo de contacto y pauta.'))

## 13. Síntesis ejecutiva automática

In [ ]:
leader_project = opportunity.index[0] if len(opportunity) else 'N/D'
leader_channel = channel.index[0] if len(channel) else 'N/D'
summary_md = f'''
### Lectura para decisión

1. El universo contiene **{len(aud):,} personas** y una tasa de avance proxy de **{aud['positiva'].mean():.1%}**.
2. La bolsa sin avance representa **{aud['negativa_sin_avance'].sum():,} personas**; no equivale automáticamente a una audiencia activable.
3. El proyecto con mayor volumen RMKT contactable es **{leader_project}**.
4. El canal con mayor escala registrada es **{leader_channel}** y el HHI es **{hhi:.3f}**.
5. Personas aptas para entrega externa: **{aptas:,}**. Si este valor es cero, la prioridad institucional es gobernanza de consentimiento, no exportación.
6. Las brechas estadísticas sirven para formular hipótesis. La decisión presupuestal debe apoyarse en lift incremental medido con grupo control y en margen, no solo en conversión bruta.
'''
display(Markdown(summary_md))

## 14. Exportación opcional de agregados

No exporta personas. Solo crea tablas agregadas para revisión o Power BI.

In [ ]:
SAVE_AGGREGATES = False
if SAVE_AGGREGATES:
    output_dir = PROJECT_ROOT / 'output' / 'marketing_audiences_lab'
    output_dir.mkdir(parents=True, exist_ok=True)
    channel.to_csv(output_dir / 'audiencias_por_canal.csv', encoding='utf-8-sig')
    opportunity.to_csv(output_dir / 'oportunidad_por_proyecto.csv', encoding='utf-8-sig')
    cohort_month.to_csv(output_dir / 'cohortes_marketing.csv', encoding='utf-8-sig')
    print(f'Agregados guardados en: {output_dir}')
else:
    print('Exportación desactivada. Cambie SAVE_AGGREGATES a True si desea CSV agregados.')